# Fink/LSST — Reload Cepheid/Pulsator Light Curves & Analysis

This notebook reloads the data saved by `02_cepheids_extended_search.ipynb`
from the `data_CEPHEIDS_DDF_02/` directory and reproduces the full analysis
(raw light curves, Lomb-Scargle period search, phase-folded diagrams,
Period-Luminosity diagram) **without any Fink API call**.

### Expected directory layout
```
data_CEPHEIDS_DDF_02/
├── df_obj.parquet
├── df_pulsators.parquet
├── df_periods.parquet        (optional — recomputed below if absent)
├── lc_dict_meta.parquet
└── lightcurves/
    └── {diaObjectId}/
        ├── sources.parquet
        └── fp.parquet
```


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- created : 2026-06-17
- last update : 2026-06-17

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import pathlib
import warnings
import os

import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle
from IPython.display import display

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("no ipympl → %matplotlib inline")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "CEPHEIDS_DDF_02"
DIR_DATA = pathlib.Path(f"data_{NB_TAG}")
LC_DIR = DIR_DATA / "lightcurves"

NB_TAG_03 = "CEPHEIDS_DDF_03"
DIR_FIGS = pathlib.Path(f"figs_{NB_TAG_03}")
DIR_FIGS.mkdir(parents=True, exist_ok=True)

assert DIR_DATA.exists(), (
    f"Data directory '{DIR_DATA}' not found.\nRun 02_cepheids_extended_search.ipynb first (section 15)."
)
print(f"Data directory : {DIR_DATA.resolve()}")
print(f"LC directory   : {LC_DIR.resolve()}")
print(f"Figs directory : {DIR_FIGS.resolve()}")

# ── Analysis parameters (must match notebook 02) ──────────────────────────────
SNR_MIN = 3.0
BANDS = list("ugrizy")
PERIOD_MIN_DAYS = 0.3
PERIOD_MAX_DAYS = 200.0
LS_SAMPLES = 20

BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 12,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(DIR_FIGS / f"{name}.{ext}", bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Helper functions

In [ ]:
def flux_to_mag(flux, flux_err, zp=31.4):
    """Convert nJy flux → AB magnitude and propagated error."""
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = zp - 2.5 * np.log10(np.where(flux > 0, flux, np.nan))
        mag_err = (2.5 / np.log(10)) * np.abs(flux_err / np.where(flux > 0, flux, np.nan))
    return mag, mag_err


def filter_lc(df: pd.DataFrame, snr_min: float = SNR_MIN) -> pd.DataFrame:
    """Apply SNR cut and add mag/mag_err columns to a sources DataFrame."""
    if df.empty:
        return df
    df = df.copy()
    flux_col = "r:psfFlux"
    err_col = "r:psfFluxErr"
    if flux_col not in df.columns or err_col not in df.columns:
        return df
    snr = np.abs(df[flux_col]) / df[err_col].replace(0, np.nan)
    df = df[snr >= snr_min].copy()
    df["mag"], df["mag_err"] = flux_to_mag(df[flux_col].values, df[err_col].values)
    df = df.dropna(subset=["mag"])
    return df.sort_values("r:midpointMjdTai").reset_index(drop=True)


def lomb_scargle_period(
    t, mag, mag_err, pmin=PERIOD_MIN_DAYS, pmax=PERIOD_MAX_DAYS, samples_per_peak=LS_SAMPLES
):
    """Return (best_period, freq_grid, power, FAP_bootstrap)."""
    if len(t) < 5:
        return np.nan, None, None, np.nan
    ls = LombScargle(t, mag, mag_err)
    fmin = 1.0 / pmax
    fmax = 1.0 / pmin
    freq, power = ls.autopower(
        minimum_frequency=fmin,
        maximum_frequency=fmax,
        samples_per_peak=samples_per_peak,
    )
    best_freq = freq[np.argmax(power)]
    best_period = 1.0 / best_freq if best_freq > 0 else np.nan
    try:
        fap = ls.false_alarm_probability(power.max(), method="bootstrap", n_bootstraps=500)
    except Exception:
        fap = np.nan
    return best_period, freq, power, fap


def phase_fold(t, period, t0=0.0):
    """Return phase in [0, 1)."""
    return ((t - t0) % period) / period


print("Helper functions defined.")

## 3. Reload summary DataFrames

In [ ]:
def _load_parquet(path: pathlib.Path, label: str) -> pd.DataFrame:
    if path.exists():
        df = pd.read_parquet(path)
        print(f"Loaded {label:25s}: {len(df):,} rows  |  cols: {list(df.columns)[:6]}...")
        return df
    print(f"NOT FOUND: {path}  ({label})")
    return pd.DataFrame()


df_obj = _load_parquet(DIR_DATA / "df_obj.parquet", "df_obj")
df_pulsators = _load_parquet(DIR_DATA / "df_pulsators.parquet", "df_pulsators")
df_periods = _load_parquet(DIR_DATA / "df_periods.parquet", "df_periods")
df_meta = _load_parquet(DIR_DATA / "lc_dict_meta.parquet", "lc_dict_meta")

print(f"\nTotal objects in survey    : {len(df_obj):,}")
print(f"Pulsating variable candidates: {len(df_pulsators):,}")
print(f"Objects with saved LC      : {len(df_meta):,}")

## 4. Reload individual light curves into `lc_dict`

In [ ]:
lc_dict = {}

if df_meta.empty:
    print("No metadata found — lc_dict will be empty.")
else:
    for _, row in df_meta.iterrows():
        oid = row["diaObjectId"]
        obj_dir = LC_DIR / str(oid)

        # Sources
        src_path = obj_dir / "sources.parquet"
        df_src = pd.read_parquet(src_path) if src_path.exists() else pd.DataFrame()

        # Forced photometry
        fp_path = obj_dir / "fp.parquet"
        df_fp = pd.read_parquet(fp_path) if fp_path.exists() else pd.DataFrame()

        # Metadata (all scalar columns except diaObjectId, n_src, n_fp)
        meta = row.drop(labels=["diaObjectId", "n_src", "n_fp"], errors="ignore").to_dict()

        lc_dict[oid] = {"src": df_src, "fp": df_fp, "meta": meta}

    print(f"Reloaded {len(lc_dict)} light curves from disk.")
    for oid, data in list(lc_dict.items())[:5]:
        print(
            f"  {oid}  src={len(data['src'])}  fp={len(data['fp'])}  "
            f"class={data['meta'].get('pulsator_class', '?')}"
        )

## 5. Summary statistics

In [ ]:
if not df_pulsators.empty:
    print("=" * 60)
    print("PULSATING VARIABLE CANDIDATES")
    print("=" * 60)
    if "pulsator_class" in df_pulsators.columns:
        print(df_pulsators["pulsator_class"].value_counts().to_string())
    if "field" in df_pulsators.columns:
        print("\nBy field:")
        print(df_pulsators["field"].value_counts().to_string())
    display(df_pulsators.head(10))
else:
    print("df_pulsators is empty.")

## 6. Raw light curves — overview plot

In [ ]:
if not lc_dict:
    print("No light curves available.")
else:
    NC_PLOT = min(50, len(lc_dict))
    ncols = 3
    nrows = int(np.ceil(NC_PLOT / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (oid, data) in enumerate(list(lc_dict.items())[:NC_PLOT]):
        ax = axes[idx]
        meta = data["meta"]
        df_filt = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()

        if df_filt.empty:
            ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        else:
            for band in BANDS:
                dfb = df_filt[df_filt["r:band"] == band]
                if dfb.empty:
                    continue
                ax.errorbar(
                    dfb["r:midpointMjdTai"],
                    dfb["mag"],
                    dfb["mag_err"],
                    fmt="o",
                    ms=2,
                    lw=0.5,
                    color=BAND_COLORS.get(band, "grey"),
                    label=band,
                    alpha=0.8,
                )
            ax.invert_yaxis()

        pclass = meta.get("pulsator_class", "?")
        stype = meta.get("f:xm_simbad_otype", "?")
        vsx = meta.get("f:xm_vsx_Type", "?")
        ax.set_title(f"{oid}\n{pclass} | SIMBAD:{stype} VSX:{vsx}", fontsize=7)
        ax.set_xlabel("MJD", fontsize=7)
        ax.set_ylabel("AB mag", fontsize=7)
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[NC_PLOT:]:
        ax.set_visible(False)

    plt.suptitle("Raw light curves — selected pulsators (reloaded)", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_raw_lc")
    plt.show()

## 7. Forced-photometry light curves

In [ ]:
objects_with_fp = {oid: data for oid, data in lc_dict.items() if not data["fp"].empty}
print(f"{len(objects_with_fp)} objects have forced-photometry data.")

if objects_with_fp:
    NC_PLOT = min(24, len(objects_with_fp))
    ncols = 3
    nrows = int(np.ceil(NC_PLOT / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (oid, data) in enumerate(list(objects_with_fp.items())[:NC_PLOT]):
        ax = axes[idx]
        meta = data["meta"]
        df_fp = data["fp"].copy()

        # Determine flux column available in FP data
        flux_col = None
        for col in ("r:psfFlux", "forcedSourceFlux", "fp_flux"):
            if col in df_fp.columns:
                flux_col = col
                break

        if flux_col is None or "r:midpointMjdTai" not in df_fp.columns:
            ax.text(
                0.5, 0.5, "No usable FP columns", ha="center", va="center", transform=ax.transAxes, fontsize=7
            )
        else:
            err_col = flux_col.replace("Flux", "FluxErr").replace("flux", "fluxErr")
            if err_col not in df_fp.columns:
                err_col = flux_col  # fallback
            band_col = "r:band" if "r:band" in df_fp.columns else None

            if band_col:
                for band in BANDS:
                    dfb = df_fp[df_fp[band_col] == band]
                    if dfb.empty:
                        continue
                    mag, mag_err = flux_to_mag(dfb[flux_col].values, dfb[err_col].values)
                    mask = np.isfinite(mag)
                    ax.errorbar(
                        dfb["r:midpointMjdTai"].values[mask],
                        mag[mask],
                        mag_err[mask],
                        fmt="s",
                        ms=2,
                        lw=0.5,
                        color=BAND_COLORS.get(band, "grey"),
                        label=band,
                        alpha=0.7,
                    )
            else:
                mag, mag_err = flux_to_mag(df_fp[flux_col].values, df_fp[err_col].values)
                mask = np.isfinite(mag)
                ax.errorbar(
                    df_fp["r:midpointMjdTai"].values[mask],
                    mag[mask],
                    mag_err[mask],
                    fmt="s",
                    ms=2,
                    color="grey",
                    alpha=0.7,
                )
            ax.invert_yaxis()

        pclass = meta.get("pulsator_class", "?")
        ax.set_title(f"{oid}\n{pclass}", fontsize=7)
        ax.set_xlabel("MJD", fontsize=7)
        ax.set_ylabel("AB mag", fontsize=7)
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[NC_PLOT:]:
        ax.set_visible(False)

    plt.suptitle("Forced-photometry light curves (reloaded)", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_fp_lc")
    plt.show()

## 8. Period search (recompute or reload)

If `df_periods.parquet` was saved by notebook 02, it is used directly.
Otherwise periods are recomputed from the reloaded sources.

In [ ]:
if df_periods.empty:
    print("df_periods not found — recomputing Lomb-Scargle periods ...")
    period_results = []

    for oid, data in lc_dict.items():
        if data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if len(df_filt) < 6:
            continue
        meta = data["meta"]

        # Prefer r-band; fall back to all bands
        df_r = df_filt[df_filt["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_filt

        best_period, _, _, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
        period_results.append(
            {
                "diaObjectId": oid,
                "field": meta.get("field", ""),
                "pulsator_class": meta.get("pulsator_class", ""),
                "simbad_otype": meta.get("f:xm_simbad_otype", ""),
                "vsx_type": meta.get("f:xm_vsx_Type", ""),
                "gcvs_type": meta.get("f:xm_gcvs_type", ""),
                "best_period_d": best_period,
                "ls_fap": fap,
                "n_pts": len(df_r),
            }
        )
        print(f"  {oid}  P={best_period:.3f}d  FAP={fap:.1e}  ({meta.get('pulsator_class', '?')})")

    df_periods = pd.DataFrame(period_results)
    if not df_periods.empty:
        df_periods.to_parquet(DIR_DATA / "df_periods.parquet", index=False)
        print(f"Saved recomputed df_periods ({len(df_periods)} rows).")
else:
    print(f"Using precomputed df_periods ({len(df_periods)} rows).")

if not df_periods.empty:
    display(df_periods.sort_values("best_period_d"))

## 9. Phase-folded light curves

In [ ]:
if df_periods.empty:
    print("No period data available.")
else:
    df_plot = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(12)
    ncols = 3
    nrows = int(np.ceil(len(df_plot) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_plot.iterrows()):
        oid = row["diaObjectId"]
        period = row["best_period_d"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if df_filt.empty:
            continue

        ax = axes[idx]
        t0 = df_filt["r:midpointMjdTai"].min()
        for band in BANDS:
            dfb = df_filt[df_filt["r:band"] == band]
            if dfb.empty:
                continue
            phi = phase_fold(dfb["r:midpointMjdTai"].values, period, t0)
            ax.errorbar(
                np.concatenate([phi, phi + 1]),
                np.concatenate([dfb["mag"].values] * 2),
                yerr=np.concatenate([dfb["mag_err"].values] * 2),
                fmt="o",
                ms=3,
                lw=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=band,
                alpha=0.85,
            )
        ax.set_xlim(0, 2)
        ax.invert_yaxis()
        ax.set_xlabel("Phase")
        ax.set_ylabel("AB mag")
        ax.set_title(
            f"{oid}  P={period:.3f}d  FAP={row['ls_fap']:.1e}\n"
            f"{row['pulsator_class']} | {row.get('vsx_type', '?')}",
            fontsize=7,
        )
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[len(df_plot) :]:
        ax.set_visible(False)

    plt.suptitle("Phase-folded light curves (sorted by LS FAP) — reloaded", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_phased_lc")
    plt.show()

## 10. Lomb-Scargle power spectrum for individual objects

In [ ]:
# Show LS periodogram for the top-N most significant objects
if not df_periods.empty:
    TOP_N = 6
    df_top = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(TOP_N)
    ncols = 2
    nrows = int(np.ceil(TOP_N / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_top.iterrows()):
        oid = row["diaObjectId"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if len(df_filt) < 5:
            continue

        df_r = df_filt[df_filt["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_filt

        _, freq, power, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
        if freq is None:
            continue

        ax = axes[idx]
        period_grid = 1.0 / freq
        ax.semilogx(period_grid, power, lw=0.8, color="steelblue")
        ax.axvline(row["best_period_d"], color="red", lw=1.5, ls="--", label=f"P={row['best_period_d']:.3f}d")
        ax.set_xlabel("Period (days)")
        ax.set_ylabel("LS power")
        ax.set_title(f"{oid}  FAP={row['ls_fap']:.1e}\n{row['pulsator_class']}", fontsize=8)
        ax.legend(fontsize=7)

    for ax in axes[TOP_N:]:
        ax.set_visible(False)

    plt.suptitle("Lomb-Scargle periodograms — top objects", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_ls_periodogram")
    plt.show()
else:
    print("No period data available.")

## 11. Period–Luminosity diagram

In [ ]:
if df_periods.empty or df_periods["best_period_d"].isna().all():
    print("No period data available.")
else:
    # Add median r-band magnitude from reloaded sources
    mag_med = []
    for oid in df_periods["diaObjectId"]:
        data = lc_dict.get(oid, {})
        if data and not data["src"].empty:
            df_r = filter_lc(data["src"])
            df_r = df_r[df_r["r:band"] == "r"]
            mag_med.append(np.nanmedian(df_r["mag"]) if len(df_r) > 0 else np.nan)
        else:
            mag_med.append(np.nan)

    df_periods = df_periods.copy()
    df_periods["mag_r_median"] = mag_med

    df_pl = df_periods.dropna(subset=["best_period_d", "mag_r_median"])
    df_pl = df_pl[(df_pl["best_period_d"] > 0.1) & (df_pl["best_period_d"] < 200)]

    class_marker = {
        "cepheid_classical": ("*", "red", 100),
        "cepheid_type2": ("^", "darkorange", 80),
        "rr_lyrae": ("o", "dodgerblue", 40),
        "delta_scuti": ("s", "green", 30),
        "lpv_mira": ("D", "purple", 40),
        "rv_tauri": ("P", "brown", 50),
        "other_pulsator": ("x", "grey", 25),
    }

    fig, ax = plt.subplots(figsize=(9, 6))
    for cls, (marker, color, size) in class_marker.items():
        sub = df_pl[df_pl["pulsator_class"] == cls]
        if sub.empty:
            continue
        ax.scatter(
            np.log10(sub["best_period_d"]),
            sub["mag_r_median"],
            marker=marker,
            color=color,
            s=size,
            alpha=0.8,
            label=f"{cls} (N={len(sub)})",
            zorder=3,
        )

    # Leavitt law reference (rough)
    logP = np.linspace(0, 2, 50)
    ax.plot(logP, -2.81 * logP + 11.5, "k--", lw=1, alpha=0.4, label="Leavitt law (rough, DM≈11)")

    ax.set_xlabel("log₁₀(Period / days)")
    ax.set_ylabel("Median r-band AB mag (apparent)")
    ax.invert_yaxis()
    ax.set_title(
        "Period–Luminosity diagram — Pulsating variables in LSST fields\n"
        "(apparent magnitudes, no distance or extinction correction)"
    )
    ax.legend(fontsize=8, loc="best")
    plt.tight_layout()
    savefig("pulsators_PL_diagram")
    plt.show()

## 12. Interactive single-object explorer

In [ ]:
# Pick any object ID from lc_dict for a detailed 4-panel view:
# top-left  : DIA sources (r:psfFlux)
# top-right : forced photometry
# bottom-left : LS periodogram
# bottom-right: phase-folded (best period)

if lc_dict:
    # Choose the object with the lowest FAP (best period detection)
    if not df_periods.empty and "ls_fap" in df_periods.columns:
        EXPLORE_OID = df_periods.dropna(subset=["ls_fap"]).sort_values("ls_fap")["diaObjectId"].iloc[0]
    else:
        EXPLORE_OID = next(iter(lc_dict))  # first available

    print(f"Exploring object: {EXPLORE_OID}")
    data = lc_dict[EXPLORE_OID]
    meta = data["meta"]
    df_src = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()
    df_fp = data["fp"]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # ── Panel 1: DIA sources ──────────────────────────────────────────────────
    ax = axes[0, 0]
    if not df_src.empty:
        for band in BANDS:
            dfb = df_src[df_src["r:band"] == band]
            if dfb.empty:
                continue
            ax.errorbar(
                dfb["r:midpointMjdTai"],
                dfb["mag"],
                dfb["mag_err"],
                fmt="o",
                ms=3,
                color=BAND_COLORS.get(band, "grey"),
                label=band,
                alpha=0.85,
            )
        ax.invert_yaxis()
    ax.set_xlabel("MJD")
    ax.set_ylabel("AB mag")
    ax.set_title(f"DIA sources — {EXPLORE_OID}")
    ax.legend(fontsize=8, ncol=3)

    # ── Panel 2: Forced photometry ────────────────────────────────────────────
    ax = axes[0, 1]
    if not df_fp.empty:
        flux_col = next((c for c in ("r:psfFlux", "forcedSourceFlux", "fp_flux") if c in df_fp.columns), None)
        if flux_col and "r:midpointMjdTai" in df_fp.columns:
            err_col = flux_col.replace("Flux", "FluxErr").replace("flux", "fluxErr")
            if err_col not in df_fp.columns:
                err_col = flux_col
            band_col = "r:band" if "r:band" in df_fp.columns else None
            if band_col:
                for band in BANDS:
                    dfb = df_fp[df_fp[band_col] == band]
                    if dfb.empty:
                        continue
                    mag, mag_err = flux_to_mag(dfb[flux_col].values, dfb[err_col].values)
                    mask = np.isfinite(mag)
                    ax.errorbar(
                        dfb["r:midpointMjdTai"].values[mask],
                        mag[mask],
                        mag_err[mask],
                        fmt="s",
                        ms=3,
                        color=BAND_COLORS.get(band, "grey"),
                        label=band,
                        alpha=0.7,
                    )
            ax.invert_yaxis()
    ax.set_xlabel("MJD")
    ax.set_ylabel("AB mag")
    ax.set_title(f"Forced photometry — {EXPLORE_OID}")
    ax.legend(fontsize=8, ncol=3)

    # ── Panel 3: LS periodogram ───────────────────────────────────────────────
    ax = axes[1, 0]
    if not df_src.empty and len(df_src) >= 5:
        df_r = df_src[df_src["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_src
        best_period, freq, power, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
        if freq is not None:
            ax.semilogx(1.0 / freq, power, lw=0.8, color="steelblue")
            if np.isfinite(best_period):
                ax.axvline(
                    best_period, color="red", lw=1.5, ls="--", label=f"P={best_period:.3f}d  FAP={fap:.1e}"
                )
            ax.legend(fontsize=8)
    ax.set_xlabel("Period (days)")
    ax.set_ylabel("LS power")
    ax.set_title("Lomb-Scargle periodogram")

    # ── Panel 4: Phase-folded ─────────────────────────────────────────────────
    ax = axes[1, 1]
    row_p = df_periods[df_periods["diaObjectId"] == EXPLORE_OID] if not df_periods.empty else pd.DataFrame()
    if not row_p.empty and np.isfinite(row_p.iloc[0]["best_period_d"]):
        P = row_p.iloc[0]["best_period_d"]
        t0 = df_src["r:midpointMjdTai"].min()
        for band in BANDS:
            dfb = df_src[df_src["r:band"] == band]
            if dfb.empty:
                continue
            phi = phase_fold(dfb["r:midpointMjdTai"].values, P, t0)
            ax.errorbar(
                np.concatenate([phi, phi + 1]),
                np.concatenate([dfb["mag"].values] * 2),
                yerr=np.concatenate([dfb["mag_err"].values] * 2),
                fmt="o",
                ms=3,
                lw=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=band,
                alpha=0.85,
            )
        ax.set_xlim(0, 2)
        ax.invert_yaxis()
        ax.set_title(f"Phase-folded  P={P:.3f}d")
        ax.legend(fontsize=8, ncol=3)
    else:
        ax.text(0.5, 0.5, "No period available", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("Phase")
    ax.set_ylabel("AB mag")

    pclass = meta.get("pulsator_class", "?")
    stype = meta.get("f:xm_simbad_otype", "?")
    vsx = meta.get("f:xm_vsx_Type", "?")
    plt.suptitle(f"{EXPLORE_OID} — {pclass} | SIMBAD:{stype} | VSX:{vsx}", fontsize=11, y=1.01)
    plt.tight_layout()
    savefig(f"explorer_{EXPLORE_OID}")
    plt.show()
else:
    print("lc_dict is empty — nothing to explore.")